# 1 First Neural Network

## Phase1: Data Loading and Alignment

- pandas: for efficient data manipulation and loading the processed Parquet/CSV files
- numpy: for high-performance numerical operations
- gc (Garbage Collector): to manually release memory and prevent RAM overflow during the heavy merging steps

In [2]:
"""
Created on Tuesday Nov 18 2025

@author: Laura Rueda
"""
import pandas as pd
import numpy as np
import gc
import os

In [5]:
# 1. SETUP
# We define the paths to the files we created in the previous notebook
CLEAN_CSV_DIR = 'data/clean_csv/'
ONE_HOT_DIR = 'data/onehotencoding/'
EMBEDDINGS_DIR = 'data/embeddings/'

os.makedirs(CLEAN_CSV_DIR, exist_ok=True)
os.makedirs(ONE_HOT_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)


print("--- STEP 1: LOADING DATA ---")

# A. Load the Base File (Patient IDs, Dates, Target)
# We only need these columns to organize the data
print("Loading Base Data (Patients & Time)...")
df_base = pd.read_csv(f'{CLEAN_CSV_DIR}deid_rx_order_final_sql.csv', 
                      usecols=['PATIENT_ID', 'Shifted_date', 'AIM_GROUP'])

# B. Load the Features (The Parquet files)
print("Loading Features (Parquet files)...")
df_meds = pd.read_parquet(f'{EMBEDDINGS_DIR}rx_code_ids.parquet')   # Embeddings Input
df_dose = pd.read_parquet(f'{ONE_HOT_DIR}dose_one_hot.parquet')  # One-Hot Input
df_route = pd.read_parquet(f'{ONE_HOT_DIR}route_one_hot.parquet') # One-Hot Input
df_freq = pd.read_parquet(f'{ONE_HOT_DIR}freq_one_hot.parquet')   # One-Hot Input

print("Data Loaded Successfully.")

# C. Verify Alignment (CRITICAL)
# All files must have exactly the same number of rows
assert len(df_base) == len(df_meds) == len(df_dose) == len(df_route) == len(df_freq)
print(f"Alignment Check Passed: All files have {len(df_base)} rows.")

--- STEP 1: LOADING DATA ---
Loading Base Data (Patients & Time)...
Loading Features (Parquet files)...
Data Loaded Successfully.
Alignment Check Passed: All files have 20436701 rows.


## Phase 2: Preparing the Target (Y)

In [6]:
print("\n--- STEP 2: ENCODING TARGET (Y) ---")

# Check unique values
print("Original Targets:", df_base['AIM_GROUP'].unique())

# Convert to Binary (0 and 1)
# 1_NoD -> 0 (Negative/Healthy)
# 2_T2D -> 1 (Positive/Diabetes)
df_base['TARGET'] = df_base['AIM_GROUP'].map({'1_NoD': 0, '2_T2D': 1})

# Check the balance
print("Target Distribution:")
print(df_base['TARGET'].value_counts())

# Drop the text column to save memory
df_base.drop(columns=['AIM_GROUP'], inplace=True)
print("Target encoded to 0/1.")


--- STEP 2: ENCODING TARGET (Y) ---
Original Targets: ['1_NoD' '2_Type2']
Target Distribution:
TARGET
0.0    16292902
Name: count, dtype: int64
Target encoded to 0/1.


## Phase 3: Merging for Sequence Creation

In [7]:
print("\n--- STEP 3: MERGING FEATURES ---")

# We concatenate (join horizontally) all our dataframes
# axis=1 means "side by side"
full_data = pd.concat([df_base, df_meds, df_dose, df_route, df_freq], axis=1)

# Inspect the result
print("Full Dataset Shape:", full_data.shape)
print(full_data.head())

# Clean up memory (Delete individual parts now that they are in 'full_data')
del df_base, df_meds, df_dose, df_route, df_freq
gc.collect()
print("Data Merged. Memory cleaned.")


--- STEP 3: MERGING FEATURES ---
Full Dataset Shape: (20436701, 97)
   PATIENT_ID         Shifted_date  TARGET  RX_CODE_ID  DOSE_0.025 mg  \
0           1  2015-10-18 15:27:00     0.0        3908              0   
1           1  2015-10-31 19:05:00     0.0        3908              0   
2           1  2016-01-10 17:03:00     0.0        4253              0   
3           1  2016-01-10 17:03:00     0.0        9928              0   
4           1  2016-01-10 17:03:00     0.0        9928              0   

   DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  DOSE_0.5 mg  ...  \
0             0            0            0            0            0  ...   
1             0            0            0            0            0  ...   
2             0            0            0            0            0  ...   
3             0            0            0            0            0  ...   
4             1            0            0            0            0  ...   

   FREQ_OTHER  FREQ_Q12H  FREQ_Q4H 

## Phase 4: Prepare Features and Labels (X and y)
This cell separates the "Metadata" (IDs, dates) from the actual "Features" (medical data) and extracts the target label for each patient

In [8]:
print("\n--- STEP 4: PREPARING FEATURE LISTS ---")

# 1. Identify columns
# We exclude patient metadata from the training features
metadata_cols = ['PATIENT_ID', 'Shifted_date', 'TARGET']
feature_cols = [c for c in full_data.columns if c not in metadata_cols]

print(f"Total Features identified: {len(feature_cols)}")
print(f"Example features: {feature_cols[:5]}")

# 2. Extract unique labels per patient (y)
# Since the TARGET is the same for every row of a patient, we take the first one.
# This results in a DataFrame with 1 row per patient.
print("Extracting unique labels per patient...")
unique_patients = full_data.drop_duplicates(subset=['PATIENT_ID'])[['PATIENT_ID', 'TARGET']]
y_data = unique_patients['TARGET'].values

print(f"Total Patients (Samples): {len(y_data)}")
print(f"Labels shape: {y_data.shape}")


--- STEP 4: PREPARING FEATURE LISTS ---
Total Features identified: 94
Example features: ['RX_CODE_ID', 'DOSE_0.025 mg', 'DOSE_0.05 mg', 'DOSE_0.1 mg', 'DOSE_0.2 mg']
Extracting unique labels per patient...
Total Patients (Samples): 248812
Labels shape: (248812,)


## Cell 5: Create 3D Sequences (Padding)
This is the most critical step. We transform the flat table into a 3D format: (Patient, Time_Step, Feature).

Strategy: We take the last 50 visits (MAX_SEQ_LEN) for each patient.

Logic: If a patient has fewer than 50 visits, we "pad" (fill) with zeros at the beginning. If they have more, we keep the most recent 50.

In [ ]:
print("\n--- STEP 5: CREATING 3D SEQUENCES (PADDING) ---")

MAX_SEQ_LEN = 50  # We look at the last 50 visits per patient
num_features = len(feature_cols)
num_patients = len(unique_patients)

# 1. Convert the entire DataFrame to a Numpy matrix (Fastest method)
# This consumes RAM, but it is necessary for speed.
print("Converting features to Numpy matrix...")
data_matrix = full_data[feature_cols].values

# 2. Indexing patient groups
# Since data is sorted, we use np.unique to find where each patient starts
print("Indexing patient groups...")
# 'start_indices' tells us the row index where each new patient begins
start_indices = np.unique(full_data['PATIENT_ID'], return_index=True)[1]

# We append the total length to know where the last patient ends
end_indices = np.append(start_indices[1:], len(full_data))

# 3. Allocate Empty 3D Memory
# Shape: (Number of Patients, 50 Time Steps, Number of Features)
print(f"Allocating 3D Memory: ({num_patients}, {MAX_SEQ_LEN}, {num_features})")
X_data = np.zeros((num_patients, MAX_SEQ_LEN, num_features), dtype='float32')

# 4. Fill the Sequences (Optimized Loop)
print("Filling sequences...")
for i, (start, end) in enumerate(zip(start_indices, end_indices)):
    # Extract all visits for THIS patient
    patient_visits = data_matrix[start:end]
    
    # Padding/Truncating Logic (Post-Padding)
    seq_len = len(patient_visits)
    
    if seq_len >= MAX_SEQ_LEN:
        # If they have more than 50 visits, take the LAST 50 (most recent)
        X_data[i, :, :] = patient_visits[-MAX_SEQ_LEN:]
    else:
        # If they have fewer (e.g., 10), fill the END of the matrix
        # [0, 0, ..., 0, visit_1, ..., visit_10] -> Preserves recent history alignment
        X_data[i, -seq_len:, :] = patient_visits

print("3D Sequences Created Successfully!")
print("Final X Shape:", X_data.shape)


--- STEP 5: CREATING 3D SEQUENCES (PADDING) ---
Converting features to Numpy matrix...
Indexing patient groups...
Allocating 3D Memory: (248812, 50, 94)
Filling sequences...
3D Sequences Created Successfully!
Final X Shape: (248812, 50, 94)


## Cell 6: Save Processed Tensors
Save the final 3D matrices (.npy format) so you can restart the kernel and train the model without reprocessing everything.

In [ ]:
print("\n--- STEP 6: SAVING READY-TO-TRAIN DATA ---")

DATA_DIR_OUTPUT = 'data/ready_for_training/'
os.makedirs(DATA_DIR_OUTPUT, exist_ok=True)

# Save X (Features) and y (Labels) as binary Numpy files
np.save(f'{DATA_DIR_OUTPUT}X_train_matrix.npy', X_data)
np.save(f'{DATA_DIR_OUTPUT}y_train_labels.npy', y_data)

# Save feature names to know what each column represents later
np.save(f'{DATA_DIR_OUTPUT}feature_names.npy', np.array(feature_cols))
print("ALL DONE! Data is ready for the Neural Network.")

# Final Cleanup
del full_data, data_matrix, X_data
gc.collect()


--- STEP 6: SAVING READY-TO-TRAIN DATA ---
🎉 ALL DONE! Data is ready for the Neural Network.


243

# Actual Deep Learning Neural Network

## Phase 7: Train/Test Split
We load the saved 3D matrices and split them. 20% of patients are kept aside for testing.

In [12]:
from sklearn.model_selection import train_test_split
import numpy as np

print("\n--- STEP 7: TRAIN/TEST SPLIT ---")

# 1. Load Data (In case you restarted the kernel)
DATA_DIR_OUTPUT = 'data/ready_for_training/'
X_data = np.load(f'{DATA_DIR_OUTPUT}X_train_matrix.npy')
y_data = np.load(f'{DATA_DIR_OUTPUT}y_train_labels.npy')
feature_names = np.load(f'{DATA_DIR_OUTPUT}feature_names.npy', allow_pickle=True)

print(f"Data Loaded. X: {X_data.shape}, y: {y_data.shape}")

# 2. Split into Train (80%) and Test (20%)
# random_state=42 ensures we get the same split every time
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples:  {X_test.shape[0]}")


--- STEP 7: TRAIN/TEST SPLIT ---
Data Loaded. X: (248812, 50, 94), y: (248812,)


ValueError: Input y contains NaN.

## Cell 8: Separate Inputs (Medication vs. Others)
We need to "slice" the 3D matrix to separate the column containing Medication IDs from the rest of the features.

In [ ]:
print("\n--- STEP 8: SEPARATING INPUTS (MEDS vs OTHERS) ---")

# 1. Locate the 'RX_CODE_ID' column
# We need to find the exact index of the medication ID feature
med_col_idx = np.where(feature_names == 'RX_CODE_ID')[0][0]
print(f"Medication IDs found at column index: {med_col_idx}")

# 2. Define Splitting Function
def split_inputs(X_matrix):
    """
    Splits the 3D tensor into two separate inputs:
    Input A: Medication IDs (for Embedding)
    Input B: All other features (for direct LSTM input)
    """
    # Extract only the Medication column (Shape: N, 50)
    X_meds = X_matrix[:, :, med_col_idx]
    
    # Extract everything else by deleting the medication column
    # (Shape: N, 50, Num_Features - 1)
    X_rest = np.delete(X_matrix, med_col_idx, axis=2)
    
    return X_meds, X_rest

# 3. Apply Split to Train and Test sets
X_train_meds, X_train_rest = split_inputs(X_train)
X_test_meds, X_test_rest = split_inputs(X_test)

print("Split Successful!")
print(f"Input A (Med IDs): {X_train_meds.shape}")
print(f"Input B (Other Features): {X_train_rest.shape}")

## Cell 9: Build the Neural Network (Keras)
Here we define the Multi-Input LSTM architecture.

Input A goes through an Embedding Layer.

Input B goes through a Masking Layer.

They merge and go into the LSTM.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

print("\n--- STEP 9: BUILDING THE LSTM MODEL ---")

# --- HYPERPARAMETERS ---
NUM_MEDS = 3500    # Approximate number of unique meds (plus buffer)
EMBED_DIM = 32     # Size of the dense vector for each medication
SEQ_LEN = 50       # Number of time steps per patient
NUM_OTHER_FEATURES = X_train_rest.shape[2]

# --- ARCHITECTURE ---

# Branch 1: Medication Input (Integers)
input_meds = Input(shape=(SEQ_LEN,), name='Input_Meds')
# Embedding Layer: Transforms IDs  into vectors of size EMBED_DIM
# mask_zero=True ignores the padding (0s) we added for short sequences
embed_layer = layers.Embedding(input_dim=NUM_MEDS, output_dim=EMBED_DIM, mask_zero=True)(input_meds)

# Branch 2: Other Features Input (One-Hot Floats)
input_rest = Input(shape=(SEQ_LEN, NUM_OTHER_FEATURES), name='Input_Rest')
# Masking Layer: Tells LSTM to ignore time steps where values are all 0
masked_rest = layers.Masking(mask_value=0.0)(input_rest)

# Merge: Combine both branches
# We concatenate them along the feature axis
merged = layers.Concatenate()([embed_layer, masked_rest])

# LSTM Layer
# 64 units: A standard size for medical time series
# dropout=0.2: Randomly drops connections to prevent overfitting
lstm_out = layers.LSTM(64, return_sequences=False, dropout=0.2)(merged)

# Output Layer
# Sigmoid activation outputs a probability between 0 and 1
output = layers.Dense(1, activation='sigmoid')(lstm_out)

# Compile Model
model = models.Model(inputs=[input_meds, input_rest], outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])

model.summary()

## Phase 10: Training
Start the training process.

In [ ]:
print("\n--- STEP 10: TRAINING ---")

# Train the model
# We pass the inputs as a list: [Meds, Rest]
history = model.fit(
    [X_train_meds, X_train_rest], y_train,
    validation_data=([X_test_meds, X_test_rest], y_test),
    epochs=10,          # Number of passes through the entire dataset
    batch_size=256,     # Number of patients processed at once
    verbose=1           # Show progress bar
)

print("Training Complete!")